# Skill fitness, one column per (method, KB)

The KB methods (`sy`, `neighbor`, `hop`, `transformer`) run once per KB: every job and CV has two
mapped skill lists, `esco_skills` and `onet_skills`, each scored on its own graph. The column name
carries the KB (`fit_skills_kb_hop_esco`, `fit_skills_kb_hop_onet`).

The two raw methods read the skill strings written in the offer and in the CV, so they have no
KB suffix.

In [ ]:
import sys; sys.path.append('../..')

import pandas as pd
from pathlib import Path
from tqdm import tqdm; tqdm.pandas()
from feature_engineering.skill_fitness import SkillFitnessCalculator

DATA_DIR = Path("../../datasets/findhr")
TRANSFORMER = "sentence-transformers/all-MiniLM-L6-v2"

KBS = {
    "esco": ("../../cache/occSkill_graph.pkl",      "esco_skills"),
    "onet": ("../../cache/onet_occSkill_graph.pkl", "onet_skills"),
}
KB_METHODS = ("sy", "neighbor", "hop", "transformer")
RAW_COLS = ("raw_skills_j", "raw_skills_c")
RAW_KB = "esco"

SKILL_COLS = [col for _, col in KBS.values()]

In [ ]:
jobs = pd.read_json(DATA_DIR / "source/filtered_jobs_dt.json").set_index("id_j")
cvs = pd.read_json(DATA_DIR / "source/filtered_cvs_dt.json").set_index("id_c")

jobs = jobs[[*SKILL_COLS, "raw_skills"]].add_suffix("_j")
cvs = cvs[[*SKILL_COLS, "raw_skills"]].add_suffix("_c")

In [ ]:
pairs = (pd.read_json(DATA_DIR / "applications/applications.json")[["id_j", "pool"]]
         .explode("pool").rename(columns={"pool": "id_c"})
         .astype({"id_c": int}).reset_index(drop=True))
pairs = (jobs
      .merge(pairs, left_index=True, right_on="id_j", how="inner", validate="one_to_many")
      .merge(cvs, left_on="id_c", right_index=True, how="inner", validate="many_to_one")
      .reset_index(drop=True))

print(len(pairs), 'pairs')
pairs.head()

In [ ]:
# One calculator per KB, freed before the next one: each holds a graph and an embedding cache.
scores = [pairs[["id_j", "id_c"]]]

for kb, (cache, skill_col) in KBS.items():
    calc = SkillFitnessCalculator(graph=Path(cache), transformer=TRANSFORMER)

    uris = (pairs[[f"{skill_col}_j", f"{skill_col}_c"]]
            .set_axis(["skills_j", "skills_c"], axis=1)
            .map(lambda labels: [*calc.graph.get_uri(labels, reduce="Down")]))
    scores.append(calc.score_dataframe(uris, fit_methods=KB_METHODS, kb=kb))

    if kb == RAW_KB:
        # The raw arm: the same cosine over the strings the ad and the CV wrote.
        scores.append(calc.score_dataframe(pairs, fit_methods=('transformer_raw',),
                                           skill_columns=RAW_COLS))
    del calc

result = pd.concat(scores, axis=1)

result["fit_skills_raw"] = pairs[list(RAW_COLS)].apply(
    axis=1, func=lambda s: SkillFitnessCalculator.exact_fit(*s)).values

In [ ]:
score_cols = [c for c in result.columns if c.startswith("fit_skills_")]
result[score_cols] = result[score_cols].round(4)
result.head()

In [ ]:
result.to_csv(DATA_DIR / 'skill_fitness.csv', index=False)